In [1]:
# !pip install -U huggingface_hub --break-system-packages

In [2]:
# !pip install --upgrade transformers --break-system-packages

In [1]:
 # !pip install "git+https://github.com/intel/auto-round.git@refs/pull/1656/head" --break-system-packages

In [3]:
!hf download Intel/gemma-4-26B-A4B-it-int4-AutoRound --local-dir ./local_model_26B

Fetching 19 files:   0%|                                 | 0/19 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Still waiting to acquire lock on local_model_26B/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on local_model_26B/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on local_model_26B/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on local_model_26B/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Fetching 19 files: 100%|████████████████████████| 19/19 [00:10<00:00,  1.74it/s]
Download complete: : 15.4GB [00:10, 2.18GB/s]              /workspace/local_model_26B
Download complete: : 15.4GB [00:10, 1.41GB/s]


In [2]:
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "./local_model_26B"
# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)

2026-04-07 16:28:00 INFO replace_modules.py L107: Experts (before replacement) [model.language_model.layers.0.experts] (Gemma4TextExperts):
Gemma4TextExperts(
  (act_fn): GELUTanh()
)
2026-04-07 16:28:00 WARNING modeling_utils.py L4432: `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
2026-04-07 16:28:00 INFO device.py L1690: Before applying custom replacements 'peak_ram': 1.1GB
2026-04-07 16:28:02 INFO moe_experts_interface.py L642: [MoE Prep] Unfused 30 MOE experts modules
2026-04-07 16:28:03 INFO device.py L1690: After applying custom replacements 'peak_ram': 1.13GB
2026-04-07 16:28:03 INFO replace_modules.py L80: Prepared 30 MOE modules for quantization
2026-04-07 16:28:03 INFO replace_modules.py L107: Experts (after replacement) [model.language_model.layers.0.experts] (Gemma4TextExperts):
Gemma4TextExperts(
  (act_fn): GELUTanh()
  (0-127): 128 x _ExpertContainer(
    (down_proj): Linear(in_features=704, out_features=2816, b

Loading weights:   0%|          | 0/35983 [00:00<?, ?it/s]

In [3]:
# Prompt - add image before text
messages = [
    {
        "role": "user", "content": [
            {"type": "image", "url": "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/Demos/sample-data/GoldenGate.png"},
            {"type": "text", "text": "What is shown in this image?"}
        ]
    }
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=512)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse output
print(processor.parse_response(response))

{'content': 'The image shows the Golden Gate Bridge in San Francisco, California. It is a large red suspension bridge extending over a body of water. In the background, there are hills and mountains. The water is calm, and there is a large rock in the foreground. On the left side of the image, there is a large building. The sky is clear and blue.', 'role': 'assistant'}
